In [1]:
"""measure_nn_runtime.py – *absolute‑RSS neural networks*

Benchmarks inference runtime for **pre‑trained Keras models** that map 5‑dim
RSS inputs → (X, Y, Z).  The stopwatch encloses:

1. `model.predict` (batch inference on the full test‑set)
2. Descaling    (inverse‑transform with `scaler_absolute.pkl`)

Nothing is written to disk – predictions are discarded after timing.  A timing
CSV can be produced if desired.

Configuration
-------------
* **Models directory** `models/nns_absolute`
  *Filename pattern*  `full_model_rss_n=400_noise=0.0_seed={seed}.h5`
* **Training filter** n = 400, noise = 0.0, category = `Measured_Normalized`.
* **Test set** `3D-Data/Measured_Normalized/gnd_n=50_noise=0.0.csv`
  (125 000 rows ⇒ 125 k forward passes per model).
* **Scaler** `scaler_absolute.pkl`.
* **Seeds scanned** 0 … 9 (only existing `.h5` files are benchmarked).
"""

from __future__ import annotations

import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model

# --------------------------------------------------------------------------------------
# Configuration ------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
MODEL_DIR = Path("models/nns_absolute")
SCALER_PATH = Path("scaler_absolute.pkl")

CATEGORY = "Measured_Normalized"
BASE_DIR = Path("3D-Data") / CATEGORY

TRAIN_N = 400     # only benchmark models trained on this sample size
NOISE = 0.0       # training + test noise
TEST_GND_N = 50   # rows in every gnd CSV (50 × 2500 = 125 000)

SEEDS = range(10)  # seeds to probe for corresponding .h5 files

SAVE_CSV = True
CSV_PATH = Path("nn_runtime_results.csv")

BATCH_SIZE = 1024  # prediction batch size; adjust if needed

# --------------------------------------------------------------------------------------
# Helper functions ---------------------------------------------------------------------
# --------------------------------------------------------------------------------------

def build_model_path(seed: int) -> Path:
    name = f"full_model_rss_n={TRAIN_N}_noise={NOISE}_seed={seed}.h5"
    return MODEL_DIR / name


def load_test_df() -> pd.DataFrame:
    fp = BASE_DIR / f"gnd_n={TEST_GND_N}_noise={NOISE}.csv"
    if not fp.exists():
        raise FileNotFoundError(f"Test CSV not found: {fp}")
    return pd.read_csv(fp)


def predict_and_descale(model, rss: np.ndarray, scaler: StandardScaler | None):
    """Run NN inference → inverse‑scale predictions to (X,Y,Z) space."""
    preds = model.predict(rss, batch_size=BATCH_SIZE, verbose=0)

    if scaler is None:
        return preds

    zeros = np.zeros_like(rss)
    descaled = scaler.inverse_transform(np.hstack([zeros, preds]))[:, -3:]
    return descaled

# --------------------------------------------------------------------------------------
# Benchmark ---------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
print("Loading resources …", flush=True)

TEST_DF = load_test_df()
N_TEST = len(TEST_DF)
RSS_TEST = TEST_DF.drop(columns=["X", "Y", "Z"]).values

SCALER = None
if SCALER_PATH.exists():
    with SCALER_PATH.open("rb") as fh:
        SCALER = pickle.load(fh)

print(
    f"Benchmarking NN models (n={TRAIN_N}, noise=0.0) on {N_TEST} samples\n"
)

timings = []

for seed in SEEDS:
    model_path = build_model_path(seed)
    if not model_path.exists():
        continue  # skip missing models

    model = load_model(model_path, compile=False)

    t0 = time.perf_counter()
    _ = predict_and_descale(model, RSS_TEST, SCALER)
    dt = time.perf_counter() - t0

    timings.append(
        {
            "seed": seed,
            "runtime_s": dt,
            "per_sample_s": dt / N_TEST,
            "n_test": N_TEST,
        }
    )

    print(
        f"Seed {seed:2d} | {N_TEST} predictions | total: {dt:.4f} s | per‑sample: {dt/N_TEST:.6f} s"
    )

if not timings:
    print("⚠️  No matching NN models found!")
else:
    results_df = pd.DataFrame(timings)
    if SAVE_CSV:
        results_df.to_csv(CSV_PATH, index=False)
        print(f"\nTiming table written to → {CSV_PATH.resolve()}")

print("\nDone.")


2025-05-14 11:05:55.804408: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-14 11:05:55.804445: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-14 11:05:55.806004: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-14 11:05:55.814902: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-14 11:05:59.340547: W tensorflow/compiler/tf2

Loading resources …


/home/sumo/anikraft/miniconda3/envs/trieste_env/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
2025-05-14 11:06:06.088331: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Benchmarking NN models (n=400, noise=0.0) on 125000 samples

Seed  0 | 125000 predictions | total: 1.3693 s | per‑sample: 0.000011 s
Seed  1 | 125000 predictions | total: 0.3229 s | per‑sample: 0.000003 s
Seed  2 | 125000 predictions | total: 0.2855 s | per‑sample: 0.000002 s
Seed  3 | 125000 predictions | total: 0.2697 s | per‑sample: 0.000002 s
Seed  4 | 125000 predictions | total: 0.3085 s | per‑sample: 0.000002 s
Seed  5 | 125000 predictions | total: 0.2707 s | per‑sample: 0.000002 s
Seed  6 | 125000 predictions | total: 0.2818 s | per‑sample: 0.000002 s
Seed  7 | 125000 predictions | total: 0.3046 s | per‑sample: 0.000002 s
Seed  8 | 125000 predictions | total: 0.2677 s | per‑sample: 0.000002 s
Seed  9 | 125000 predictions | total: 0.2668 s | per‑sample: 0.000002 s

Timing table written to → /home/sumo/anikraft/RSS/chapter4_recreating_results/nn_runtime_results.csv

Done.


In [2]:
"""measure_nn_runtime.py – *relative‑RSS neural networks*

Benchmarks inference runtime for **pre‑trained Keras models** that consume the
10‑dim *relative* RSS feature vector (e.g. RSS0/RSS1, …) and predict (X, Y, Z).
The stopwatch encloses:

1. `model.predict` (batch inference on the full test‑set)
2. Descaling    (inverse‑transform with `scaler_relative.pkl`)

Predictions are discarded – **nothing is saved** apart from an optional timings
CSV.

Configuration
-------------
* **Models directory** `models/nns_relative`
  *Filename pattern*  `full_model_relative_rss_n=400_noise=0.0_seed={seed}.h5`
* **Training filter** n = 400, noise = 0.0, category = `Measured_relative_Normalized`.
* **Test set** `3D-Data/Measured_relative_Normalized/relative_gnd_n=50_noise=0.0.csv`
  (125 000 rows ⇒ 125 k forward passes per model).
* **Scaler** `scaler_relative.pkl`.
* **Seeds scanned** 0 … 9 (only existing `.h5` files are benchmarked).
"""

from __future__ import annotations

import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model

# --------------------------------------------------------------------------------------
# Configuration ------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
MODEL_DIR = Path("models/nns_relative")
SCALER_PATH = Path("scaler_relative.pkl")

CATEGORY = "Measured_relative_Normalized"
BASE_DIR = Path("3D-Data") / CATEGORY

TRAIN_N = 400     # only benchmark models trained on this sample size
NOISE = 0.0       # training + test noise
TEST_GND_N = 50   # rows in every gnd CSV (50 × 2500 = 125 000)

SEEDS = range(10)  # seeds to probe for corresponding .h5 files

SAVE_CSV = True
CSV_PATH = Path("nn_runtime_results_relative.csv")

BATCH_SIZE = 1024  # prediction batch size; adjust if needed

# --------------------------------------------------------------------------------------
# Helper functions ---------------------------------------------------------------------
# --------------------------------------------------------------------------------------

def build_model_path(seed: int) -> Path:
    name = f"full_model_relative_rss_n={TRAIN_N}_noise={NOISE}_seed={seed}.h5"
    return MODEL_DIR / name


def load_test_df() -> pd.DataFrame:
    fp = BASE_DIR / f"relative_gnd_n={TEST_GND_N}_noise={NOISE}.csv"
    if not fp.exists():
        raise FileNotFoundError(f"Test CSV not found: {fp}")
    return pd.read_csv(fp)


def predict_and_descale(model, rss: np.ndarray, scaler: StandardScaler | None):
    """Run NN inference → inverse‑scale predictions to (X,Y,Z) space."""
    preds = model.predict(rss, batch_size=BATCH_SIZE, verbose=0)

    if scaler is None:
        return preds

    zeros = np.zeros_like(rss)
    descaled = scaler.inverse_transform(np.hstack([zeros, preds]))[:, -3:]
    return descaled

# --------------------------------------------------------------------------------------
# Benchmark ---------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
print("Loading resources …", flush=True)

TEST_DF = load_test_df()
N_TEST = len(TEST_DF)
RSS_TEST = TEST_DF.drop(columns=["X", "Y", "Z"]).values

SCALER = None
if SCALER_PATH.exists():
    with SCALER_PATH.open("rb") as fh:
        SCALER = pickle.load(fh)

print(
    f"Benchmarking *relative‑RSS* NN models (n={TRAIN_N}, noise=0.0) on {N_TEST} samples\n"
)

timings = []

for seed in SEEDS:
    model_path = build_model_path(seed)
    if not model_path.exists():
        continue  # skip missing models

    model = load_model(model_path, compile=False)

    t0 = time.perf_counter()
    _ = predict_and_descale(model, RSS_TEST, SCALER)
    dt = time.perf_counter() - t0

    timings.append(
        {
            "seed": seed,
            "runtime_s": dt,
            "per_sample_s": dt / N_TEST,
            "n_test": N_TEST,
        }
    )

    print(
        f"Seed {seed:2d} | {N_TEST} predictions | total: {dt:.4f} s | per‑sample: {dt/N_TEST:.6f} s"
    )

if not timings:
    print("⚠️  No matching NN models found!")
else:
    results_df = pd.DataFrame(timings)
    if SAVE_CSV:
        results_df.to_csv(CSV_PATH, index=False)
        print(f"\nTiming table written to → {CSV_PATH.resolve()}")

print("\nDone.")


Loading resources …


/home/sumo/anikraft/miniconda3/envs/trieste_env/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Benchmarking *relative‑RSS* NN models (n=400, noise=0.0) on 125000 samples

Seed  0 | 125000 predictions | total: 0.3108 s | per‑sample: 0.000002 s
Seed  1 | 125000 predictions | total: 0.2807 s | per‑sample: 0.000002 s
Seed  2 | 125000 predictions | total: 0.2822 s | per‑sample: 0.000002 s
Seed  3 | 125000 predictions | total: 0.3001 s | per‑sample: 0.000002 s
Seed  4 | 125000 predictions | total: 0.2780 s | per‑sample: 0.000002 s
Seed  5 | 125000 predictions | total: 0.3091 s | per‑sample: 0.000002 s
Seed  6 | 125000 predictions | total: 0.2931 s | per‑sample: 0.000002 s
Seed  7 | 125000 predictions | total: 0.3164 s | per‑sample: 0.000003 s
Seed  8 | 125000 predictions | total: 0.2978 s | per‑sample: 0.000002 s
Seed  9 | 125000 predictions | total: 0.2631 s | per‑sample: 0.000002 s

Timing table written to → /home/sumo/anikraft/RSS/chapter4_recreating_results/nn_runtime_results_relative.csv

Done.


In [3]:
"""measure_nn_runtime.py – *log‑distance neural networks*

Benchmarks end‑to‑end inference runtime for **pre‑trained Keras models** that:

1. Predict five **log‑distances** (D0…D4) from 5‑dim RSS inputs.
2. Inverse‑scale those predictions with `scaler_log_d.pkl`.
3. Exponentiate → linear metres.
4. Perform non‑linear least‑squares trilateration to recover (X, Y, Z).

The stopwatch encloses **all four stages**.  Predictions are discarded; only
an optional timing CSV is written.

Configuration
-------------
* **Models directory** `models/nns_log_d`
  *Filename pattern*  `full_model_rss_n=400_noise=0.0_seed={seed}.h5`
* **Training filter** n = 400, noise = 0.0, category =`Measured_log_d_Normalized`.
* **Test set** `3D-Data/Measured_log_d_Normalized/gnd_n=50_noise=0.0.csv`
  (125 000 rows ⇒ 125 k forward passes + trilaterations per model).
* **Scaler** `scaler_log_d.pkl`.
* **Seeds scanned** 0 … 9 (only existing `.h5` files are benchmarked).
"""

from __future__ import annotations

import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model

# --------------------------------------------------------------------------------------
# Configuration ------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
MODEL_DIR = Path("models/nns_log_d")
SCALER_PATH = Path("scaler_log_d.pkl")

CATEGORY = "Measured_log_d_Normalized"
BASE_DIR = Path("3D-Data") / CATEGORY

TRAIN_N = 400     # only benchmark models trained on this sample size
NOISE = 0.0       # training + test noise
TEST_GND_N = 50   # rows in every gnd CSV (50 × 2500 = 125 000)

#SEEDS = range(10)
SEEDS = range(1)
SAVE_CSV = True
CSV_PATH = Path("nn_runtime_results_logd.csv")

BATCH_SIZE = 1024  # batch size for model.predict

# Beacon coordinates for trilateration
BEACON_COORDS = np.array(
    [
        [0, 0, 5],
        [-2, -2, 5],
        [-2,  2, 5],
        [2, -2, 5],
        [2,  2, 5],
    ]
)

# --------------------------------------------------------------------------------------
# Trilateration helpers ----------------------------------------------------------------
# --------------------------------------------------------------------------------------

def _mse_error(pos, beacon_coords, distances):
    pred = np.linalg.norm(beacon_coords - pos, axis=1)
    return np.mean((pred - distances) ** 2)


def trilaterate_mse(distances, beacon_coords=BEACON_COORDS):
    guess = np.mean(beacon_coords, axis=0)
    bounds = [(None, None), (None, None), (None, 5)]  # constrain z
    res = minimize(_mse_error, guess, args=(beacon_coords, distances), method="L-BFGS-B", bounds=bounds)
    if not res.success:
        raise ValueError("Trilateration failed: " + res.message)
    return res.x

# --------------------------------------------------------------------------------------
# Helper functions ---------------------------------------------------------------------
# --------------------------------------------------------------------------------------

def build_model_path(seed: int) -> Path:
    name = f"full_model_rss_n={TRAIN_N}_noise={NOISE}_seed={seed}.h5"
    return MODEL_DIR / name


def load_test_df() -> pd.DataFrame:
    fp = BASE_DIR / f"gnd_n={TEST_GND_N}_noise={NOISE}.csv"
    if not fp.exists():
        raise FileNotFoundError(f"Test CSV not found: {fp}")
    return pd.read_csv(fp)


DIST_COLS = ["D0", "D1", "D2", "D3", "D4"]


def full_pipeline(model, rss: np.ndarray, scaler: StandardScaler | None):
    """Predict log‑distances → inverse‑scale → exp → trilaterate → XYZ array."""
    # 1) NN prediction (log‑distances)
    preds = model.predict(rss, batch_size=BATCH_SIZE, verbose=0)

    # 2) Descale if scaler provided
    if scaler is not None:
        zeros = np.zeros_like(rss)
        preds = scaler.inverse_transform(np.hstack([zeros, preds]))[:, -5:]

    # 3) Linear metres
    lin_d = np.exp(preds)

    # 4) Trilaterate each sample
    coords = np.empty((lin_d.shape[0], 3))
    for i, d in enumerate(lin_d):
        coords[i] = trilaterate_mse(d)

    return coords

# --------------------------------------------------------------------------------------
# Benchmark ---------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
print("Loading resources …", flush=True)

TEST_DF = load_test_df()
N_TEST = len(TEST_DF)
RSS_TEST = TEST_DF.drop(columns=DIST_COLS).values  # inputs are RSS0..RSS4

SCALER = None
if SCALER_PATH.exists():
    with SCALER_PATH.open("rb") as fh:
        SCALER = pickle.load(fh)

print(
    f"Benchmarking *log‑d* NN models (n={TRAIN_N}, noise=0.0) on {N_TEST} samples\n"
)

timings = []

for seed in SEEDS:
    model_path = build_model_path(seed)
    if not model_path.exists():
        continue

    model = load_model(model_path, compile=False)

    start = time.perf_counter()
    _ = full_pipeline(model, RSS_TEST, SCALER)
    dt = time.perf_counter() - start

    timings.append(
        {
            "seed": seed,
            "runtime_s": dt,
            "per_sample_s": dt / N_TEST,
            "n_test": N_TEST,
        }
    )

    print(
        f"Seed {seed:2d} | {N_TEST} predictions | total: {dt:.2f} s | per‑sample: {dt/N_TEST:.6f} s"
    )

if not timings:
    print("⚠️  No matching log‑d NN models found!")
else:
    df_results = pd.DataFrame(timings)
    if SAVE_CSV:
        df_results.to_csv(CSV_PATH, index=False)
        print(f"\nTiming table written to → {CSV_PATH.resolve()}")

print("\nDone.")


Loading resources …


/home/sumo/anikraft/miniconda3/envs/trieste_env/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Benchmarking *log‑d* NN models (n=400, noise=0.0) on 125000 samples

Seed  0 | 125000 predictions | total: 676.37 s | per‑sample: 0.005411 s

Timing table written to → /home/sumo/anikraft/RSS/chapter4_recreating_results/nn_runtime_results_logd.csv

Done.
